#### Generate continous conditions from metadata

In [7]:
import json

mwir_parent_dir = "/mnt/data/ataparia/darpa/mwir dataset unseen/val/"
mwir_file_path = f"{mwir_parent_dir}metadata.jsonl"
vis_parent_dir = "/mnt/data/ataparia/darpa/visible dataset unseen/val/"
vis_file_path = f"{vis_parent_dir}metadata.jsonl"

with open(vis_file_path, "r") as file:
    lines = file.readlines()
vis_data = []
for line in lines:
    vis_data.append(json.dumps(json.loads(line)))

with open(mwir_file_path, "r") as file:
    lines = file.readlines()
mwir_data = []
for line in lines:
    mwir_data.append(json.dumps(json.loads(line)))

In [8]:
from PIL import Image
import os
import numpy as np
from tqdm import tqdm

human_conditions = {}

for vis_line, ir_line in tqdm(zip(vis_data, mwir_data), total=len(vis_data), desc="Processing data"):
    vis = json.loads(vis_line)
    ir  = json.loads(ir_line)
    if vis["image_id"] != ir["image_id"]:
        print("Image ID mismatch:", vis["image_id"], ir["image_id"])
        continue

    img_id = vis["image_id"]

    # 1) continuous range in [0,1]
    rng = vis["objects"]["range"][0]
    norm_range = rng / 5000.0

    # 2) continuous aspect angle in [0,1]
    ang = vis["objects"]["aspect_angle"][0]  # 0–360
    norm_angle = ang / 360.0

    # 3) intensity comparison as 0/1
    def mean_intensity(path):
        im = np.array(Image.open(path).convert("L"), dtype=np.float32) / 255.0
        return im[im>0].mean()

    ir_path  = os.path.join(mwir_parent_dir,  ir["file_name"])
    vis_path = os.path.join(vis_parent_dir, vis["file_name"])
    flag_num = int(mean_intensity(ir_path) > mean_intensity(vis_path))

    # assemble numeric conditions
    human_conditions[img_id] = [norm_range, norm_angle, flag_num]

Processing data: 100%|██████████| 11937/11937 [02:33<00:00, 77.62it/s]


In [9]:
# Save the human conditions to a JSON file
with open("conditions/unseen/continous_test.json", "w") as file:
    json.dump(human_conditions, file, indent=4)
print("Continous conditions saved!")

Continous conditions saved!


### Generate human conditions from metadata

In [ ]:
import json

mwir_file_path = "/mnt/data/ataparia/darpa/mwir dataset unseen/test/metadata.jsonl"
vis_file_path = "/mnt/data/ataparia/darpa/visible dataset unseen/test/metadata.jsonl"

with open(vis_file_path, "r") as file:
    lines = file.readlines()
vis_data = []
for line in lines:
    vis_data.append(json.dumps(json.loads(line)))

with open(mwir_file_path, "r") as file:
    lines = file.readlines()
mwir_data = []
for line in lines:
    mwir_data.append(json.dumps(json.loads(line)))

In [ ]:
# Conditions:
# 1. Range Conditions
#     - Range is between 0 and 1000.
#     - Range is between 1000 and 2000.
#     - Range is between 2000 and 3000.
#     - Range is between 3000 and 4000.
#     - Range is between 4000 and 5000.
# 2. Aspect Angle Conditions
#     - Aspect angle between 0 and 45.
#     - Aspect angle between 45 and 90.
#     - Aspect angle between 90 and 135.
#     - Aspect angle between 135 and 180.
#     - Aspect angle between 180 and 225.
#     - Aspect angle between 225 and 270.
#     - Aspect angle between 270 and 315.
#     - Aspect angle between 315 and 360.
# 3. Intensity Comparison Condition
#     - MWIR intensity is greater than visible intensity.

from PIL import Image
import os
import numpy as np
from tqdm import tqdm

human_conditions = {}

for vis_info, ir_info in tqdm(zip(vis_data, mwir_data), total=len(vis_data), desc="Processing data"):
    vis_info = json.loads(vis_info)
    ir_info = json.loads(ir_info)
    if vis_info["image_id"] == ir_info["image_id"]:
        img_id = vis_info["image_id"]
        range_ = vis_info["objects"]["range"][0]
        aspect_angle = vis_info["objects"]["aspect_angle"][0]
        
        condition = []
        
        # Range Conditions
        if 0 <= range_ < 1000:
            condition.extend([True, False, False, False, False])
        elif 1000 <= range_ < 2000:
            condition.extend([False, True, False, False, False])
        elif 2000 <= range_ < 3000:
            condition.extend([False, False, True, False, False])
        elif 3000 <= range_ < 4000:
            condition.extend([False, False, False, True, False])
        elif 4000 <= range_ <= 5000:
            condition.extend([False, False, False, False, True])
        
        # Aspect Angle Conditions
        if 0 <= aspect_angle < 45:
            condition.extend([True, False, False, False, False, False, False, False])
        elif 45 <= aspect_angle < 90:
            condition.extend([False, True, False, False, False, False, False, False])
        elif 90 <= aspect_angle < 135:
            condition.extend([False, False, True, False, False, False, False, False])
        elif 135 <= aspect_angle < 180:
            condition.extend([False, False, False, True, False, False, False, False])
        elif 180 <= aspect_angle < 225:
            condition.extend([False, False, False, False, True, False, False, False])
        elif 225 <= aspect_angle < 270:
            condition.extend([False, False, False, False, False, True, False, False])
        elif 270 <= aspect_angle < 315:
            condition.extend([False, False, False, False, False, False, True, False])
        elif 315 <= aspect_angle <= 360:
            condition.extend([False, False, False, False, False, False, False, True])
        
        # Intensity comparison condition by reading the image
        ir_image = ir_info["file_name"]
        ir_image = Image.open(os.path.join("/mnt/data/ataparia/darpa/mwir dataset unseen/test", ir_image))
        ir_image = np.array(ir_image)
        ir_image = ir_image.astype(np.float32)
        ir_image = ir_image / 255.0
        ir_image = ir_image[ir_image > 0]
        ir_pixel_intensity = ir_image.mean()
        
        vis_image = vis_info["file_name"]
        vis_image = Image.open(os.path.join("/mnt/data/ataparia/darpa/visible dataset unseen/test", vis_image))
        vis_image = np.array(vis_image)
        vis_image = vis_image.astype(np.float32)
        vis_image = vis_image / 255.0
        vis_image = vis_image[vis_image > 0]
        vis_pixel_intensity = vis_image.mean()
        if ir_pixel_intensity > vis_pixel_intensity:
            condition.append(True)
            # print("MWIR intensity is greater than visible intensity")
            # print("MWIR pixel intensity:", ir_pixel_intensity)
            # print("Visible pixel intensity:", vis_pixel_intensity)
            # print("Image ID:", img_id)
        else:
            condition.append(False)
        human_conditions[img_id] = condition

    else:
        print("Image ID mismatch:", vis_info["image_id"], ir_info["image_id"])

In [ ]:
# Save the human conditions to a JSON file
with open("conditions/unseen/human_test.json", "w") as file:
    json.dump(human_conditions, file, indent=4)
print("Human conditions saved!")

### Generate application specific conditions

In [ ]:
# import json

# # Read the saved JSON file
# with open("conditions/unseen/vlm_test.json", "r") as file:
#     human_conditions = json.load(file)

# print("Length of human conditions:", len(human_conditions))

# # Read refine conditions
# questions_file = 'conditions/preprocess/refined_conditions.json'

# with open(questions_file, 'r') as f:
#     questions_list = json.load(f)

# length = len(questions_list)
# print("Length of questions_list:", length)

# # Check the length of each human condition
# count = 0

# for key, value in human_conditions.items():
#     if len(value) != length:
#         count += 1
#         print(f"Image ID: {key}, Length of human condition: {len(value)}")

# print("Count of human conditions with length not equal to questions_list:", count)

Count of human conditions with length not equal to questions_list: 0


In [1]:
from PIL import Image
import numpy as np
import torch
import re
import json
import os

In [2]:
import json
parent_path = "/mnt/data/ataparia/darpa/visible dataset unseen/test"
vis_file_path = f"{parent_path}/metadata.jsonl"

with open(vis_file_path, "r") as file:
    lines = file.readlines()
vis_data = []
for line in lines:
    vis_data.append(json.dumps(json.loads(line)))

In [3]:
questions_file = 'conditions/preprocess/refined_conditions.json'

with open(questions_file, 'r') as f:
    questions_list = json.load(f)
    
# formatted_questions = "\n".join(["- " + question for question in questions_list])
formatted_questions = "\n".join([f"{i+1}. {question}" for i, question in enumerate(questions_list)])

In [4]:
from openai import OpenAI
import base64
from io import BytesIO

In [5]:
import os
client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY")
)

In [6]:
def get_conditions(image, retry_count=0, max_retries=3):   
    if hasattr(image, "detach"):
        image = image.detach().cpu().numpy()
        if image.ndim == 3 and image.shape[0] in [1, 3]:
            image = np.transpose(image, (1, 2, 0))
        if image.dtype in [np.float32, np.float64]:
            image = (image * 255).astype(np.uint8)

    if isinstance(image, np.ndarray):
        if image.ndim == 3 and image.shape[0] in [1, 3]:
            image = np.transpose(image, (1, 2, 0))
        try:
            image = Image.fromarray(image)
        except Exception as e:
            raise ValueError(
                "Failed to convert numpy array to PIL image. Ensure the array has the correct shape and dtype."
            ) from e
    
    def encode_image(image):
        buffer = BytesIO()
        image.save(buffer, format="JPEG")
        buffer.seek(0) 
        return base64.b64encode(buffer.read()).decode("utf-8")

    image_encoded = encode_image(image)
    conditions = []
    
    message_text = (
        "Answer the following questions based on the given image:\n"
        "## Questions:\n"
        f"{formatted_questions}\n\n"
        f"IMPORTANT: Your answer must be a JSON object with exactly {len(questions_list)} keys. "
        f"The keys should be the numbers from 1 to {len(questions_list)} (as strings) and each value must be a boolean (True or False), one for each question, and nothing else. "
        "The image is provided after these questions."
    )
    
    response = client.chat.completions.create(
        # model="gpt-4o",
        model="gpt-4o-2024-11-20",
        messages=[
            {
            "role": "system",
            "content": (
                    "You are a highly specialized assistant that provides concise answers to specific questions about images. "
                    "For each question, respond with either True or False only. "
                    "Provide your answer as a JSON object with keys corresponding to the question numbers (from 1 to "
                    f"{len(questions_list)}). Do not provide additional context or descriptions."
                )
            },
            {
            "role": "user",
            "content": [
                {
                "type": "text",
                "text": message_text,
                },
                {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{image_encoded}",
                },
                },
            ],
            }
        ],
        )
    
    if response.choices[0].message.content is None:
        return get_conditions(image, retry_count=retry_count, max_retries=max_retries)
    
    response_text = response.choices[0].message.content.strip()
    
    response_text = re.sub(r"```(json)?", "", response_text).strip()
    response_text = re.sub(r"```", "", response_text).strip()
    try:
        conditions = json.loads(response_text)
    except json.JSONDecodeError:
        conditions_list = [item.strip() for item in response_text.strip("[]").split(",")]
        conditions = {str(i+1): (True if str(item).strip().lower() == "true" else False)
                      for i, item in enumerate(conditions_list)}
    
    if not isinstance(conditions, dict) or len(conditions.keys()) != len(questions_list):
        if retry_count < max_retries:
            print(f"Mismatch in response structure (got {len(conditions) if isinstance(conditions, dict) else 'non-dict'} vs expected {len(questions_list)}). Retrying {retry_count+1}/{max_retries}...")
            time.sleep(0.5)
            return get_conditions(image, retry_count=retry_count+1, max_retries=max_retries)
        else:
            # raise ValueError("Response structure does not match the number of questions even after retries.")
            print("Response structure does not match the number of questions even after retries.")
            return []
    
    sorted_conditions = [conditions[str(i+1)] for i in range(len(questions_list))]
    return sorted_conditions

In [7]:
conditions = {}

In [ ]:
import time

def process_image(data):
    image_id = data["image_id"]
    image_path = data["file_name"]
    image = Image.open(os.path.join(parent_path, image_path)).convert('RGB')
    condition = get_conditions(image)
    time.sleep(0.25)
    
    return image_id, condition

In [9]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

with ThreadPoolExecutor(max_workers=32) as executor:
    futures = []
    for data in tqdm(vis_data, desc="Processing images"):
        data = json.loads(data)
        future = executor.submit(process_image, data)
        futures.append(future)

    for future in tqdm(futures, desc="Collecting results"):
        image_id, condition = future.result()
        conditions[image_id] = condition

Processing images: 100%|██████████| 11952/11952 [00:00<00:00, 32060.42it/s]


In [10]:
# Save the conditions to a JSON file
with open("conditions/unseen/vlm_test.json", "w") as file:
    json.dump(conditions, file, indent=4)